# Baselines and evaluation

## 1. Split and metrics
I split matches in time: train 2000-2021, validation 2022-2023, test from 2024. I only look at the test set once, at the end.
I score probabilities with log loss, Brier score and ranked probability score (RPS). Lower is better for all three.

In [2]:
import sys
from pathlib import Path

sys.path.append(str(Path.cwd().parent))

import numpy as np

from src.evaluation import OUTCOMES, compare
from src.splits import load_matches

matches = load_matches()
train = matches[matches["split"] == "train"]
valid = matches[matches["split"] == "valid"]
matches["split"].value_counts()

split
train    19490
test      2583
valid     1887
Name: count, dtype: int64

## 2. Naive baselines
I estimate outcome frequencies on the training set only, then apply them to the validation set.

In [3]:
freq_overall = train["result"].value_counts(normalize=True)[OUTCOMES].to_numpy()
freq_by_venue = train.groupby("neutral")["result"].value_counts(normalize=True).unstack()[OUTCOMES]

predictions = {
    "uniform": np.full((len(valid), 3), 1 / 3),
    "overall_frequency": np.tile(freq_overall, (len(valid), 1)),
    "frequency_by_venue": freq_by_venue.reindex(valid["neutral"]).to_numpy(),
}
compare(predictions, valid["result"])

,log_loss,brier,rps
uniform,1.0986,0.6667,0.2399
overall_frequency,1.0490,0.6324,0.2283
frequency_by_venue,1.0436,0.6283,0.2263


The uniform baseline gives a log loss of ln(3) = 1.0986, the floor any model must beat.
Using outcome frequencies by venue improves all three metrics: the model gives on average 35.2% to the actual outcome (exp(-1.0436)), against 33.3% for uniform. This is the score to beat.